In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip -q install sentence-transformers

In [3]:
import pandas as pd
import numpy as np

from tqdm.auto import tqdm

from sentence_transformers import CrossEncoder

In [4]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

sample_submission = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print(train.shape)
print(test.shape)

train.head()

(2000, 8)
(500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [5]:
MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

print("Loading Cross Encoder...")

cross_encoder = CrossEncoder(MODEL_NAME)

print("Loaded!")

Loading Cross Encoder...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Loaded!


In [6]:
def build_prompt(row):

    return f"""You are an expert at solving multiple choice questions.

Read the question carefully.

Question:
{row["prompt"]}

Evaluate the following candidate answer.
"""

In [7]:
OPTION_LETTERS = ["A","B","C","D","E"]

def predict_top3(row):

    prompt = build_prompt(row)

    pairs = []

    for letter in OPTION_LETTERS:

        option = str(row[letter])

        pairs.append([
            prompt,
            option
        ])

    scores = cross_encoder.predict(
        pairs,
        show_progress_bar=False
    )

    ranking = np.argsort(scores)[::-1]

    letters = [
        OPTION_LETTERS[i]
        for i in ranking
    ]

    return " ".join(letters[:3])

In [8]:
print(predict_top3(train.iloc[0]))

D B A


In [9]:
predictions = []

for _, row in tqdm(test.iterrows(), total=len(test)):

    predictions.append(
        predict_top3(row)
    )

  0%|          | 0/500 [00:00<?, ?it/s]

In [10]:
submission = pd.DataFrame({

    "ID":test["id"],

    "Prediction":predictions

})

submission.head()

,ID,Prediction
0,1,A C D
1,2,C B A
2,3,D C A
3,4,C E A
4,5,C A D


In [11]:
submission.to_csv(

    "submission.csv",

    index=False

)

print("Saved submission.csv")

submission.head()

Saved submission.csv


,ID,Prediction
0,1,A C D
1,2,C B A
2,3,D C A
3,4,C E A
4,5,C A D


In [12]:
def map3(actual, pred):

    pred = pred.split()

    for i, p in enumerate(pred):

        if p == actual:

            return 1/(i+1)

    return 0


train_predictions = []

for _, row in tqdm(train.iterrows(), total=len(train)):

    train_predictions.append(
        predict_top3(row)
    )

score = np.mean([
    map3(a,p)
    for a,p in zip(train["answer"], train_predictions)
])

print("Train MAP@3 =", round(score,4))

  0%|          | 0/2000 [00:00<?, ?it/s]

Train MAP@3 = 0.4311
